Preparacion de las muestras nuevas(son del mismo dataset)

In [0]:
import os, shutil

# Volumen para el streaming
spark.sql("CREATE VOLUME IF NOT EXISTS proyecto.sentimiento.stream_vol")
ruta_stream = "/Volumes/proyecto/sentimiento/stream_vol/entrada/"
ruta_checkpoint = "/Volumes/proyecto/sentimiento/stream_vol/checkpoint/"

# Limpiar de corridas previas
for r in [ruta_stream, ruta_checkpoint]:
    if os.path.exists(r):
        shutil.rmtree(r)
os.makedirs(ruta_stream)

# Apartar 200 reseñas reales del dataset como "nuevas"
muestra = (spark.table("proyecto.sentimiento.silver_reviews")
    .select("Id", "Text")
    .limit(200))

# Convertir a lista de Python para el simulador
resenas_nuevas = [(r["Id"], r["Text"]) for r in muestra.collect()]
print(f"Reseñas reales apartadas para simular ingesta: {len(resenas_nuevas)}")

Reseñas reales apartadas para simular ingesta: 200


ReadStream + cargar el modelo

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, when

# Cargar el modelo ya entrenado
modelo = PipelineModel.load("/Volumes/proyecto/sentimiento/datos/modelo_sentimiento")

# Esquema de las reseñas entrantes
esquema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("Text", StringType(), True)
])

# Stream que escucha la carpeta
df_stream = (spark.readStream
    .schema(esquema)
    .option("header", "false")
    .option("sep", "\t")
    .csv(ruta_stream))

# CLASIFICAR en vivo con el modelo
df_clasificado = modelo.transform(df_stream).withColumn("sentimiento_predicho",
    when(col("prediction") == 0, "Negativo")
    .when(col("prediction") == 1, "Neutro")
    .otherwise("Positivo"))

# Nos quedamos con lo relevante
df_salida = df_clasificado.select("Id", "sentimiento_predicho")

WriteStream a memoria

In [0]:
query = (df_salida.writeStream
    .format("memory")
    .queryName("clasificacion_vivo")
    .outputMode("append")           # append: cada reseña clasificada se agrega
    .option("checkpointLocation", ruta_checkpoint)
    .trigger(availableNow=True)
    .start())

print("Stream de clasificación configurado.")

Stream de clasificación configurado.


Simulador + dashboard en vivo

In [0]:
import time
from IPython.display import clear_output

# PASO 1: soltar las reseñas reales en la carpeta (en lotes de 20)
print("Inyectando reseñas reales en la carpeta...")
for i in range(0, len(resenas_nuevas), 20):
    lote = resenas_nuevas[i:i+20]
    lineas = []
    for id_r, texto in lote:
        # limpiar tabs/saltos del texto para que no rompan el CSV
        texto_limpio = str(texto).replace("\t", " ").replace("\n", " ")
        lineas.append(f"{id_r}\t{texto_limpio}\n")
    with open(f"{ruta_stream}lote_{i}.csv", "w") as f:
        f.writelines(lineas)
print(f"{len(resenas_nuevas)} reseñas inyectadas.\n")

# PASO 2: arrancar el stream (procesa lo que hay)
query = (df_salida.writeStream
    .format("memory").queryName("clasificacion_vivo")
    .outputMode("append")
    .option("checkpointLocation", ruta_checkpoint)
    .trigger(availableNow=True).start())

# PASO 3: dashboard en vivo
for k in range(10):
    clear_output(wait=True)
    print(f"CLASIFICACIÓN DE RESEÑAS EN VIVO - Actualización {k+1}")

    print("\n--- Últimas reseñas clasificadas ---")
    try:
        spark.sql("SELECT Id, sentimiento_predicho FROM clasificacion_vivo").show(8, truncate=False)
    except:
        print("Esperando...")

    print("--- Conteo por sentimiento (en vivo) ---")
    try:
        spark.sql("""SELECT sentimiento_predicho, COUNT(*) as cantidad
                     FROM clasificacion_vivo GROUP BY sentimiento_predicho
                     ORDER BY cantidad DESC""").show()
    except:
        print("Esperando...")

    if not query.isActive:
        break
    time.sleep(2)

query.awaitTermination()
print("\nClasificación en vivo finalizada.")
spark.sql("""SELECT sentimiento_predicho, COUNT(*) as cantidad
             FROM clasificacion_vivo GROUP BY sentimiento_predicho ORDER BY cantidad DESC""").show()

CLASIFICACIÓN DE RESEÑAS EN VIVO - Actualización 3

--- Últimas reseñas clasificadas ---
+---+--------------------+
|Id |sentimiento_predicho|
+---+--------------------+
|81 |Positivo            |
|82 |Positivo            |
|83 |Neutro              |
|84 |Neutro              |
|85 |Neutro              |
|86 |Positivo            |
|87 |Positivo            |
|88 |Positivo            |
+---+--------------------+
only showing top 8 rows
--- Conteo por sentimiento (en vivo) ---
+--------------------+--------+
|sentimiento_predicho|cantidad|
+--------------------+--------+
|            Positivo|     140|
|              Neutro|      31|
|            Negativo|      29|
+--------------------+--------+


Clasificación en vivo finalizada.
+--------------------+--------+
|sentimiento_predicho|cantidad|
+--------------------+--------+
|            Positivo|     140|
|              Neutro|      31|
|            Negativo|      29|
+--------------------+--------+



In [0]:
import shutil, os

ruta_checkpoint = "/Volumes/proyecto/sentimiento/stream_vol/checkpoint/"
if os.path.exists(ruta_checkpoint):
    shutil.rmtree(ruta_checkpoint)
print("Checkpoint limpiado, listo para arrancar de nuevo.")

Checkpoint limpiado, listo para arrancar de nuevo.
